<p align="center"><a href="https://www.plus2net.com/python/file-seek.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Python file.seek(): Stream Pointer Positioning & Whence Modes Practice

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/file-seek.php](https://www.plus2net.com/python/file-seek.php)

Run cells in order in Google Colab. All random file access concepts, `whence` reference points (`os.SEEK_SET`, `os.SEEK_CUR`, `os.SEEK_END`), text mode constraints vs binary mode flexibility, tail reading of large files, in-place byte patching with mode `'r+b'`, and self-checking assertions are embedded directly in this notebook. Save a copy in Google Drive to keep your edits.

### 1. Setup: Creating Sample Binary and Text Files
Generate testing payloads with known byte sequences.

In [ ]:
# Create binary sample with known 36-byte payload
sample_payload = b"0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
with open("sample_seek.bin", "wb") as f:
    f.write(sample_payload)

# Create text sample
with open("sample_seek.txt", "w", encoding="utf-8") as f:
    f.write("Line 1: Python\nLine 2: Stream\nLine 3: Cursor\n")

print("Sample files created successfully.")


### 2. Seeking from File Start (SEEK_SET = 0)
`f.seek(offset, os.SEEK_SET)` moves the cursor to an absolute offset from the beginning of the file.

In [ ]:
import os

with open("sample_seek.bin", "rb") as f:
    f.seek(10, os.SEEK_SET)  # Move to byte 10
    read_bytes = f.read(5)
    print("Bytes read at offset 10:", read_bytes)  # b'ABCDE'
    print("New cursor position:", f.tell())         # 15


### 3. Relative Seeking from Current Cursor (SEEK_CUR = 1)
`f.seek(offset, os.SEEK_CUR)` moves the cursor relative to its current location in binary mode.

In [ ]:
with open("sample_seek.bin", "rb") as f:
    f.seek(10)               # Cursor at 10
    f.seek(5, os.SEEK_CUR)   # Advance 5 more bytes -> offset 15
    chunk = f.read(5)
    print("Bytes read from relative offset:", chunk)  # b'FGHIJ'
    print("Cursor now at:", f.tell())                 # 20


### 4. Seeking Backwards from End of File (SEEK_END = 2)
`f.seek(-offset, os.SEEK_END)` allows reading the tail or trailer of files without reading all earlier bytes.

In [ ]:
with open("sample_seek.bin", "rb") as f:
    # Seek 6 bytes backward from the end
    f.seek(-6, os.SEEK_END)
    tail_bytes = f.read(6)
    print("Last 6 bytes of file:", tail_bytes)  # b'UVWXYZ'


### 5. Multi-Pass Text Stream Processing via seek(0) Rewind
Rewind file streams with `f.seek(0)` to read multiple passes without reopening files.

In [ ]:
with open("sample_seek.txt", "r", encoding="utf-8") as stream:
    # Pass 1: Count total lines
    line_count = sum(1 for _ in stream)
    print(f"Pass 1: Total lines counted = {line_count}")
    
    # Rewind back to start
    stream.seek(0)
    
    # Pass 2: Ingest header line
    first_line = stream.readline().strip()
    print(f"Pass 2: Header = {first_line}")


### 6. Text Mode Restriction Demonstration
In text mode, relative seeks with non-zero offsets raise `io.UnsupportedOperation`.

In [ ]:
import io

with open("sample_seek.txt", "r", encoding="utf-8") as text_file:
    try:
        text_file.seek(5, 1)  # Invalid in text mode
    except io.UnsupportedOperation as err:
        print("Caught expected text mode restriction error:", err)


### 7. Self-Checking Verification Suite
Run unit assertions verifying `seek()` navigation across all whence reference points.

In [ ]:
class SeekVerifier:
    @staticmethod
    def read_span(path: str, offset: int, length: int) -> bytes:
        with open(path, "rb") as stream:
            stream.seek(offset, os.SEEK_SET)
            return stream.read(length)

v = SeekVerifier()
assert v.read_span("sample_seek.bin", 10, 5) == b"ABCDE"

with open("sample_seek.bin", "rb") as f:
    f.seek(-4, os.SEEK_END)
    assert f.read(4) == b"WXYZ"
    
    f.seek(10)
    f.seek(5, os.SEEK_CUR)
    assert f.read(5) == b"FGHIJ"

print("All file.seek() unit assertions passed successfully! 100% verified.")
